# 02 – Embedding & Indexing

This notebook demonstrates the full **Phase 2** pipeline:

1. Load, clean, and chunk 3 sample PDFs (Phase 1).
2. Embed all chunks with **BAAI/bge-small-en-v1.5**.
3. Upsert into a **Qdrant** collection (make sure `docker compose up -d` is running).
4. Visualise the embedding space with **UMAP** (2-D, coloured by source document).
5. Run 5 sample queries and print top-3 results with scores.
6. Analyse score distributions with a histogram.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
_root = Path().resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f"Project root: {_root}")

## 1 · Ingest & Chunk Sample PDFs

In [ ]:
from src.ingestion.loader import DocumentLoader
from src.ingestion.cleaner import DocumentCleaner
from src.ingestion.chunker import RecursiveChunker
from src.config.settings import get_settings

cfg = get_settings()
loader  = DocumentLoader()
cleaner = DocumentCleaner()
chunker = RecursiveChunker(cfg)

data_dir = _root / "data" / "sample_docs"
pdf_files = sorted(data_dir.glob("*.pdf"))[:3]
print(f"Found {len(pdf_files)} PDF(s): {[p.name for p in pdf_files]}")

all_docs = []
for pdf in pdf_files:
    docs = loader.load_file(pdf)
    all_docs.extend(cleaner.clean_batch(docs))

chunks = chunker.chunk_documents(all_docs)
print(f"\nTotal chunks produced: {len(chunks)}")
print(f"Sample chunk:\n  id      : {chunks[0].chunk_id}")
print(f"  tokens  : {chunks[0].token_count}")
print(f"  content : {chunks[0].content[:80]}…")

## 2 · Embed Chunks

In [ ]:
from src.embedding.bge_embedder import BGEEmbedder

embedder = BGEEmbedder(
    model_name=cfg.embedding_model,
    normalize=True,
    batch_size=cfg.embedding_batch_size,
)

print(f"Model    : {embedder.model_name}")
print(f"Dimension: {embedder.dimension}")

texts = [c.content for c in chunks]
vectors = embedder.embed_batch(texts, mode="passage")

print(f"\nEmbedded {len(vectors)} chunks")
print(f"Vector shape: ({len(vectors)}, {len(vectors[0])})")

## 3 · Index into Qdrant

> Make sure Qdrant is running first: `docker compose up -d`

In [ ]:
from src.vectorstore.qdrant_store import QdrantStore
from src.indexing.pipeline import IndexingPipeline

COLLECTION = "ragforge_notebook"

store = QdrantStore(
    host=cfg.qdrant_host,
    port=cfg.qdrant_port,
    collection_name=COLLECTION,
)

if not store.health_check():
    raise RuntimeError(
        "Qdrant is not reachable. Run: docker compose up -d"
    )
print("✓ Qdrant is healthy")

# Delete collection if it exists so we start fresh each run.
try:
    store.delete_collection(COLLECTION)
    print(f"✓ Dropped existing collection '{COLLECTION}'")
except Exception:
    pass

pipeline = IndexingPipeline(
    embedder=embedder,
    vector_store=store,
    collection_name=COLLECTION,
    auto_create_collection=True,
)

report = pipeline.run(chunks)
print(report)

## 4 · Qdrant Collection Stats

In [ ]:
import json
info = store.collection_info(COLLECTION)
print(json.dumps(info, indent=2, default=str))

## 5 · UMAP Embedding Space Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

X = np.array(vectors, dtype=np.float32)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_2d = reducer.fit_transform(X)

# Colour by source document.
source_paths = [c.metadata.get("source_path", "unknown") for c in chunks]
unique_sources = sorted(set(source_paths))
source_to_idx = {s: i for i, s in enumerate(unique_sources)}
colours = [source_to_idx[s] for s in source_paths]

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=colours, cmap="tab10", alpha=0.7, s=30, linewidths=0,
)

handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=plt.cm.tab10(i / max(len(unique_sources), 1)),
               markersize=8, label=Path(s).name)
    for i, s in enumerate(unique_sources)
]
ax.legend(handles=handles, title="Source document", loc="best", fontsize=8)
ax.set_title("UMAP projection of BGE embeddings (coloured by source PDF)", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("umap_embeddings.png", dpi=150)
plt.show()
print("UMAP plot saved → umap_embeddings.png")

## 6 · Sample Queries — Top-3 Results with Scores

In [ ]:
QUERIES = [
    "What is retrieval-augmented generation?",
    "How does document chunking affect retrieval quality?",
    "What embedding models are used for semantic search?",
    "Explain the HNSW index algorithm.",
    "What are the benefits of using vector databases?",
]

all_scores = []

for q in QUERIES:
    q_vec = embedder.embed_single(q, mode="query")
    results = store.search(q_vec, top_k=3)
    all_scores.extend([r.score for r in results])

    print(f"\n🔍 Query: {q!r}")
    print(f"  {'Rank':<5} {'Score':>7}  {'Snippet':<60}")
    print(f"  {'-'*5} {'-'*7}  {'-'*60}")
    for r in results:
        snippet = r.content.replace("\n", " ")[:60]
        print(f"  {r.rank:<5} {r.score:>7.4f}  {snippet}")

## 7 · Score Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

scores_arr = np.array(all_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(scores_arr, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(scores_arr.mean(), color="crimson", linestyle="--", label=f"Mean={scores_arr.mean():.3f}")
axes[0].set_title("Similarity Score Distribution")
axes[0].set_xlabel("Cosine Similarity Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Box plot per query
per_query = [all_scores[i*3:(i+1)*3] for i in range(len(QUERIES))]
axes[1].boxplot(per_query, labels=[f"Q{i+1}" for i in range(len(QUERIES))], patch_artist=True)
axes[1].set_title("Score Distribution per Query")
axes[1].set_xlabel("Query")
axes[1].set_ylabel("Cosine Similarity Score")

plt.tight_layout()
plt.savefig("score_distribution.png", dpi=150)
plt.show()

print(f"\n📊 Score Statistics")
print(f"   Min   : {scores_arr.min():.4f}")
print(f"   Max   : {scores_arr.max():.4f}")
print(f"   Mean  : {scores_arr.mean():.4f}")
print(f"   Std   : {scores_arr.std():.4f}")
print("\n   Saved → score_distribution.png")